 # Chapter 57 — Document Chunking: Breaking a PDF into Retrievable Pieces

Project
In this chapter, you will take a real PDF file, extract its text page by page, and split that text into smaller overlapping chunks. Each chunk keeps its source filename, page number, chunk ID, and word positions so a future retrieval system can point back to the source.

We will use the illustrative five-page Transaction Monitoring Alert Case Summary PDF from the earlier PDF-processing work. The information in that PDF is synthetic and for learning only.
Learning outcomes
By the end, you will be able to:
- extract selectable text from a PDF page by page;
- inspect page-level extraction quality before chunking;
- implement a configurable word-based chunker with overlap;
- preserve page-level provenance in every chunk;
- validate chunk size, overlap, IDs, and source metadata;
- export the resulting chunks to JSONL and CSV.

# Install the dependency 

In [1]:
%pip install -q pypdf pandas

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# 2 Import Libraries 

In [2]:
from pathlib import Path
import hashlib
import json
import re
import pandas as pd
from pypdf import PdfReader

# 3. Locate the PDF you downloaded

In [3]:
PDF_NAME = "Chapter 57 - Alert_Case_Summary.pdf"
PDF_PATH = Path(PDF_NAME)

In [4]:
if not PDF_PATH.is_file():
    pdf_candidates = sorted(Path.cwd().glob("*.pdf"))
    if len(pdf_candidates) == 1:
        PDF_PATH = pdf_candidates[0]
        print(f"expected file name not found")
    elif len(pdf_candidates) == 0:
        raise FileNotFoundError("File not found, update pdf path")

    else: 
        raise FileNotFoundError("More than one pdf was found")



 PDF_PATH: A pathlib.Path object pointing to the target PDF file.

 .read_bytes(): Opens the file in binary mode ("rb"), reads its entire content into memory as raw bytes (bytes object), and automatically closes the file.

 pdf_bytes: Holds the binary data of the PDF.

In [5]:
pdf_bytes = PDF_PATH.read_bytes()

In [6]:
pdf_bytes

b'%PDF-1.4\n%\x93\x8c\x8b\x9e ReportLab Generated PDF document (opensource)\n1 0 obj\n<<\n/F1 2 0 R /F2 3 0 R\n>>\nendobj\n2 0 obj\n<<\n/BaseFont /Helvetica /Encoding /WinAnsiEncoding /Name /F1 /Subtype /Type1 /Type /Font\n>>\nendobj\n3 0 obj\n<<\n/BaseFont /Helvetica-Bold /Encoding /WinAnsiEncoding /Name /F2 /Subtype /Type1 /Type /Font\n>>\nendobj\n4 0 obj\n<<\n/Contents 20 0 R /MediaBox [ 0 0 595.2756 841.8898 ] /Parent 19 0 R /Resources <<\n/Font 1 0 R /ProcSet [ /PDF /Text /ImageB /ImageC /ImageI ]\n>> /Rotate 0 /Trans <<\n\n>> \n  /Type /Page\n>>\nendobj\n5 0 obj\n<<\n/Contents 21 0 R /MediaBox [ 0 0 595.2756 841.8898 ] /Parent 19 0 R /Resources <<\n/Font 1 0 R /ProcSet [ /PDF /Text /ImageB /ImageC /ImageI ]\n>> /Rotate 0 /Trans <<\n\n>> \n  /Type /Page\n>>\nendobj\n6 0 obj\n<<\n/BitsPerComponent 8 /ColorSpace /DeviceRGB /Filter [ /ASCII85Decode /FlateDecode ] /Height 415 /Length 31510 /SMask 7 0 R \n  /Subtype /Image /Type /XObject /Width 1146\n>>\nstream\nGb",k$(+]2q=SB1(hR/X"qY

In [7]:
pdf_sha256 = hashlib.sha256(pdf_bytes).hexdigest()

hashlib.sha256(pdf_bytes): Feeds the raw bytes into the SHA-256 hashing algorithm to initialize a hash object.

.hexdigest(): Returns the resulting 256-bit hash digest as a 64-character string of lowercase hexadecimal digits (0–9, a–f).

Purpose: Generates a content-based fingerprint. If even a single byte of the PDF changes, this string will change completely.

In [8]:
pdf_sha256

'16b4413cee36cff56f0c6194ce069486f84abfbb0aad7db4e6b5913d671d5673'

In [9]:
document_id = f"PDF-{pdf_sha256[:12].upper()}"


In [10]:
document_id

'PDF-16B4413CEE36'

# 4 Inspect the PDF before extracting text

A PDF has pages, but those pages do not guarantee that the words are stored as selectable text. First inspect basic metadata and page count.

In [11]:
reader = PdfReader(str(PDF_PATH))

In [12]:
reader

In [13]:
PDF_PATH.name

'Chapter 57 - Alert_Case_Summary.pdf'

In [14]:
len(reader.pages)

5

In [15]:
reader.metadata

{'/Author': '(anonymous)',
 '/CreationDate': "D:20261019024030+00'00'",
 '/Creator': '(unspecified)',
 '/Keywords': '',
 '/ModDate': "D:20261019024030+00'00'",
 '/Producer': 'ReportLab PDF Library - (opensource)',
 '/Subject': '(unspecified)',
 '/Title': '(anonymous)',
 '/Trapped': '/False'}

# 5. Extract text page by page

We keep each page separate because it gives us an easy, auditable source location. Page numbers in this notebook are one-based: the first page is page 1, not page 0.

In [16]:
page_records = []

for page_index, page in enumerate(reader.pages):
    page_number = page_index + 1
    extracted_text = page.extract_text() or ""
    page_records.append(extracted_text)

In [17]:
page_records

['Illustrative synthetic data — not a real customer or bank investigation\nPage 1 of 5\nTRANSACTION MONITORING\nALERT CASE SUMMARY\nA practical introduction to machine learning, alerts, cases and cross-border transaction analysis\n1. What is Machine Learning (ML)?\nMachine Learning is a branch of artificial intelligence in which a system learns patterns from data to make predictions, rank risks or\nidentify unusual behaviour. Instead of relying only on manually written rules, an ML model can learn what typical activity looks like\nfrom historical examples and flag transactions that differ from expected patterns.\nML concept\nMeaning in transaction monitoring\nFeatures\nInputs such as amount, currency, country, customer type, transaction frequency and prior behaviour.\nModel / score\nA model estimates risk or unusualness; a high score is a signal to review, not proof of wrongdoing.\nAlert\nA rule or model output that highlights activity for review by an analyst.\nHuman investigation\nAn

In [18]:
 # Another way to do append  more details in dictionary format

page_records = []

for page_index, page in enumerate(reader.pages):
    page_number = page_index + 1
    extracted_text = page.extract_text() or ""
    words = extracted_text.split()
    page_records.append({
        "document_id" : document_id,
        "source_file" : PDF_PATH.name,
        "page_number" : page_number,
        "text" : extracted_text,
        "character_count" : len(extracted_text),
        "extraction_status" : "text_extracted" if words else " no selectable test",
    })

In [19]:
pages_df = pd.DataFrame(page_records)

In [20]:
pages_df

,document_id,source_file,page_number,text,character_count,extraction_status
0,PDF-16B4413CEE36,Chapter 57 - Alert_Case_Summary.pdf,1,Illustrative synthetic data — not a real custo...,1592,text_extracted
1,PDF-16B4413CEE36,Chapter 57 - Alert_Case_Summary.pdf,2,FINANCIAL INTELLIGENCE | TRANSACTION MONITORIN...,1821,text_extracted
2,PDF-16B4413CEE36,Chapter 57 - Alert_Case_Summary.pdf,3,FINANCIAL INTELLIGENCE | TRANSACTION MONITORIN...,1485,text_extracted
3,PDF-16B4413CEE36,Chapter 57 - Alert_Case_Summary.pdf,4,FINANCIAL INTELLIGENCE | TRANSACTION MONITORIN...,1213,text_extracted
4,PDF-16B4413CEE36,Chapter 57 - Alert_Case_Summary.pdf,5,FINANCIAL INTELLIGENCE | TRANSACTION MONITORIN...,2159,text_extracted


 # 6 . What exactly is a chunk 

A chunk is a smaller text segment cut from a page's extracted text. In this chapter we count whitespace-separated words, rather than model tokens, because it makes the algorithm easy to inspect.

This first implementation intentionally does not allow a chunk to cross a page boundary. That makes page citations straightforward. Later systems can use larger document-level or section-aware chunks when appropriate.

Field	Meaning

document_id	= ID derived from the PDF's SHA-256 hash

source_file	= Original PDF filename

page_number	= One-based page number in the PDF

chunk_id = 	Unique ID for one chunk

chunk_index = 	One-based order within that page

word_start =	Zero-based position of the first word within the page text

word_end_exclusive	= Position just after the last word within the page text

word_count =	Number of words in this chunk

text =	Text passed to a future embedding/retrieval step

# 7. Implement a word-based chunker

The chunker uses a sliding window. For example, chunk_size=120 and overlap=20 means a chunk contains up to 120 words, and the next chunk starts 100 words later, repeating 20 words from the previous chunk.
The overlap is useful because a sentence or rule can straddle a chunk boundary. Too much overlap, however, creates extra duplicate text.

In [22]:
page_records

[{'document_id': 'PDF-16B4413CEE36',
  'source_file': 'Chapter 57 - Alert_Case_Summary.pdf',
  'page_number': 1,
  'text': 'Illustrative synthetic data — not a real customer or bank investigation\nPage 1 of 5\nTRANSACTION MONITORING\nALERT CASE SUMMARY\nA practical introduction to machine learning, alerts, cases and cross-border transaction analysis\n1. What is Machine Learning (ML)?\nMachine Learning is a branch of artificial intelligence in which a system learns patterns from data to make predictions, rank risks or\nidentify unusual behaviour. Instead of relying only on manually written rules, an ML model can learn what typical activity looks like\nfrom historical examples and flag transactions that differ from expected patterns.\nML concept\nMeaning in transaction monitoring\nFeatures\nInputs such as amount, currency, country, customer type, transaction frequency and prior behaviour.\nModel / score\nA model estimates risk or unusualness; a high score is a signal to review, not proof

In [23]:
page_records

[{'document_id': 'PDF-16B4413CEE36',
  'source_file': 'Chapter 57 - Alert_Case_Summary.pdf',
  'page_number': 1,
  'text': 'Illustrative synthetic data — not a real customer or bank investigation\nPage 1 of 5\nTRANSACTION MONITORING\nALERT CASE SUMMARY\nA practical introduction to machine learning, alerts, cases and cross-border transaction analysis\n1. What is Machine Learning (ML)?\nMachine Learning is a branch of artificial intelligence in which a system learns patterns from data to make predictions, rank risks or\nidentify unusual behaviour. Instead of relying only on manually written rules, an ML model can learn what typical activity looks like\nfrom historical examples and flag transactions that differ from expected patterns.\nML concept\nMeaning in transaction monitoring\nFeatures\nInputs such as amount, currency, country, customer type, transaction frequency and prior behaviour.\nModel / score\nA model estimates risk or unusualness; a high score is a signal to review, not proof

# 9. Implement a word-based chunker

In [ ]:
def chunk_page_text(page_records, chunk_size=120, overlap=20):
    # 1. Type and value validations
    if not isinstance(page_records, list):
        raise TypeError("page_records must be a list of page dictionaries")
    if not isinstance(chunk_size, int) or chunk_size <= 0:
        raise ValueError("chunk_size must be a positive integer")
    if not isinstance(overlap, int) or overlap < 0:
        raise ValueError("overlap must be a non-negative integer")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    all_chunks = []
    step = chunk_size - overlap

    for page_record in page_records:
        if not isinstance(page_record, dict):
            raise TypeError("Each element in page_records must be a dictionary")

        text = page_record.get("text", "")
        if not isinstance(text, str):
            raise TypeError("page_record['text'] must be a string")

        words = text.split()
        if not words:
            continue

        start = 0
        chunk_index = 1
        page_number = int(page_record["page_number"])

        while start < len(words):
            end = min(start + chunk_size, len(words))
            all_chunks.append({
                "document_id": page_record["document_id"],
                "source_file": page_record["source_file"],
                "page_number": page_number,
                "chunk_id": f"{page_record['document_id']}-P{page_number:03d}-C{chunk_index:03d}",
                "chunk_index": chunk_index,
                "word_start": start,
                "word_count": end - start,
                "text": " ".join(words[start:end]),
            })

            if end == len(words):
                break

            start += step
            chunk_index += 1

    return all_chunks

In [39]:
chunks = chunk_page_text(page_records, chunk_size=120, overlap=20)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 13


# 14. Try a keyword lookup over the chunks

In [40]:
def keyword_search_chunks(query, chunks, max_results =5):
    if not isinstance(query, str):
        raise ValueError("query must be a non empty string")
    if not isinstance(max_results, int) or max_results < 1:
        raise ValueError("max results mist be a positive integer")

    query_lower = query.casefold()

    results = []

    for record in chunks:
        if query_lower in record["text"].casefold():
            results.append({
                "source_file": record["source_file"],
                "page_number": record["page_number"],
                "chunk_id": record["chunk_id"],
                "text": record["text"],
            })
            if len(results) >= max_results:
                break
    return results

search_results = keyword_search_chunks("How to read the numbers", chunks, max_results=5)
pd.DataFrame(search_results)

,source_file,page_number,chunk_id,text
0,Chapter 57 - Alert_Case_Summary.pdf,2,PDF-16B4413CEE36-P002-C002,suspicious activity reporting No Further Actio...


In [41]:
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(search_results))

,source_file,page_number,chunk_id,text
0,Chapter 57 - Alert_Case_Summary.pdf,2,PDF-16B4413CEE36-P002-C002,"suspicious activity reporting No Further Action (NFA) 277 Cases closed without further escalation in this example Average investigation time 4.2 days Illustrative average from opening to closure Transactions in example case 18 Transactions linked to the selected case review Customers / entities linked 3 One customer and two related counterparties/entities How to read the numbers Alerts and cases are not interchangeable: multiple alerts can be grouped into one case, and not every alert becomes a case. SAR and NFA are example dispositions and must be applied according to jurisdictional requirements and internal procedures. The sample totals are illustrative, not operational benchmarks. Case definition checklist Case element What should be recorded Identity Case ID, customer ID, customer type, industry, account /"
